# Battery Storage Modeling
This notebook defines the battery dispatch algorithm and evaluates storage performance across the three renewable energy scenarios.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from dataclasses import dataclass

project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

scenario_path = processed_dir / 'scenario_forecasts.csv'
scenarios_df = pd.read_csv(scenario_path)
scenarios_df['date'] = pd.to_datetime(scenarios_df['date'])

In [ ]:
@dataclass
class BatteryConfig:
    capacity_gwh: float
    charge_rate_gw: float
    discharge_rate_gw: float
    round_trip_efficiency: float
    min_soc_pct: float
    max_soc_pct: float
    capex_per_mwh: float
    annual_opex_pct: float


def calculate_battery_dispatch(scenario_df: pd.DataFrame, battery_config: BatteryConfig) -> pd.DataFrame:
    capacity_mwh = battery_config.capacity_gwh * 1000
    charge_rate_mw = battery_config.charge_rate_gw * 1000
    discharge_rate_mw = battery_config.discharge_rate_gw * 1000
    efficiency = battery_config.round_trip_efficiency
    min_soc = capacity_mwh * battery_config.min_soc_pct / 100
    max_soc = capacity_mwh * battery_config.max_soc_pct / 100
    current_soc = capacity_mwh * 0.5

    rows = []
    for _, row in scenario_df.iterrows():
        load = row['load_mw']
        renewable = row['renewable_total_mw']
        shortfall = row['shortfall_mw']
        surplus = row['surplus_mw']
        charge = 0.0
        discharge = 0.0
        unmet = shortfall

        if surplus > 0:
            max_charge = min(surplus, charge_rate_mw, (max_soc - current_soc) / efficiency)
            charge = max(0.0, max_charge)
            current_soc += charge * efficiency
        elif shortfall > 0:
            max_discharge = min(shortfall, discharge_rate_mw, current_soc)
            discharge = max(0.0, max_discharge)
            current_soc -= discharge
            unmet = max(0.0, shortfall - discharge)

        current_soc = np.clip(current_soc, min_soc, max_soc)
        rows.append({
            'date': row['date'],
            'load_mw': load,
            'renewable_total_mw': renewable,
            'shortfall_mw': shortfall,
            'surplus_mw': surplus,
            'battery_charge_mw': charge,
            'battery_discharge_mw': discharge,
            'battery_soc_mwh': current_soc,
            'battery_soc_pct': current_soc / capacity_mwh * 100,
            'unmet_demand_mw': unmet,
        })

    return pd.DataFrame(rows)

In [ ]:
configs = {
    'Battery_5GWh': BatteryConfig(5.0, 1.0, 1.0, 0.87, 20.0, 100.0, 150, 2.5),
    'Battery_10GWh': BatteryConfig(10.0, 2.0, 2.0, 0.87, 20.0, 100.0, 150, 2.5),
    'Battery_20GWh': BatteryConfig(20.0, 3.0, 3.0, 0.87, 20.0, 100.0, 150, 2.5),
    'Battery_50GWh': BatteryConfig(50.0, 5.0, 5.0, 0.87, 20.0, 100.0, 150, 2.5),
}

results = []
for scenario_name in ['worst', 'average', 'best']:
    scenario_data = scenarios_df[scenarios_df['scenario'] == scenario_name].reset_index(drop=True)
    for config_name, config in configs.items():
        dispatch_df = calculate_battery_dispatch(scenario_data, config)
        shortfall_reduction = ((scenario_data['shortfall_mw'].sum() - dispatch_df['unmet_demand_mw'].sum()) / scenario_data['shortfall_mw'].sum() * 100) if scenario_data['shortfall_mw'].sum() > 0 else 0
        results.append({
            'scenario': scenario_name,
            'battery_config': config_name,
            'capacity_gwh': config.capacity_gwh,
            'shortfall_reduction_pct': shortfall_reduction,
            'unmet_days': (dispatch_df['unmet_demand_mw'] > 0).sum(),
            'avg_soc_pct': dispatch_df['battery_soc_pct'].mean(),
            'residual_shortfall_gwh': dispatch_df['unmet_demand_mw'].sum() / 1000 / 24,
        })
        dispatch_df.to_csv(processed_dir / f'dispatch_{scenario_name}_{config_name.lower().replace("_", "")}.csv', index=False)

summary_df = pd.DataFrame(results)
summary_df.to_csv(processed_dir / 'battery_storage_analysis.csv', index=False)

recommendations = []
for scenario_name in ['worst', 'average', 'best']:
    row = summary_df[summary_df['scenario'] == scenario_name].sort_values('shortfall_reduction_pct', ascending=False).iloc[0]
    recommendations.append({
        'scenario': scenario_name,
        'recommended_battery_config': row['battery_config'],
        'battery_capacity_gwh': row['capacity_gwh'],
        'shortfall_reduction_pct': row['shortfall_reduction_pct'],
        'residual_shortfall_gwh': row['residual_shortfall_gwh'],
    })

pd.DataFrame(recommendations).to_csv(processed_dir / 'battery_recommendations.csv', index=False)
print(f"Saved battery_storage_analysis.csv and battery_recommendations.csv to {processed_dir}")
summary_df.head()